# 01. LLM 기초 — 메시지, 프롬프트, 스트리밍

에이전트를 만들기 전에, LLM과 대화하는 기본 방법을 익힙니다.

## 학습 목표

- 메시지의 역할(`system`, `human`, `ai`)을 이해한다
- 시스템 메시지로 모델의 행동을 제어한다
- `model.stream()`으로 실시간 응답을 받는다

## 1.1 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-5.4-mini")
print("\u2713 모델 준비 완료")

✓ 모델 준비 완료


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 1.2 메시지의 세 가지 역할

LLM은 **메시지 리스트**를 입력으로 받습니다. 각 메시지에는 역할이 있으며, 역할(role), 콘텐츠(content), 메타데이터(metadata) 세 가지 핵심 요소로 구성됩니다.

| 역할 | 클래스 | 설명 |
|---|---|---|
| `system` | `SystemMessage` | 모델의 행동 지침을 설정합니다. 페르소나, 응답 톤, 규칙 등을 정의하여 모델의 초기 동작을 결정합니다. |
| `human` | `HumanMessage` | 사용자의 입력을 나타냅니다. 텍스트뿐만 아니라 이미지, 오디오, 파일 등 멀티모달 콘텐츠를 지원합니다. |
| `ai` | `AIMessage` | 모델의 응답입니다. 텍스트 응답 외에도 `tool_calls`(도구 호출), `usage_metadata`(토큰 사용량) 등의 속성을 포함합니다. |

이 외에도 도구 실행 결과를 모델에 전달하는 `ToolMessage`가 있습니다. `ToolMessage`는 도구 결과 내용, 도구 호출 ID, 도구 이름을 필수로 포함합니다.

In [3]:
from langchain.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="당신은 유용한 어시스턴트입니다."),
    HumanMessage(content="Python이란 무엇인가요?"),
]

response = model.invoke(messages, config=lf_config)
print("응답:", response.content[:200])

응답: Python(파이썬)은 **사람이 읽기 쉬운 문법**을 가진 **고급 프로그래밍 언어**입니다.

### 특징
- **문법이 간단함**: 초보자도 배우기 쉽습니다.
- **다용도**:  
  - 웹 개발  
  - 데이터 분석  
  - 인공지능/머신러닝  
  - 자동화  
  - 게임 제작 등
- **풍부한 라이브러리**: 필요한 기능을 쉽게 추가할 수


## 1.3 시스템 메시지로 행동 제어

`SystemMessage`를 바꾸면 같은 질문에도 전혀 다른 응답을 받습니다.
이것이 **프롬프트 엔지니어링**의 핵심입니다.

In [4]:
# 같은 질문, 다른 시스템 메시지
question = HumanMessage(content="중력에 대해 설명해 주세요.")

# 과학자 페르소나
r1 = model.invoke([SystemMessage(content="당신은 물리학자입니다. 정확하게 답하세요."), question], config=lf_config)
print("[과학자]", r1.content[:150])
print()

# 어린이 교사 페르소나
r2 = model.invoke([SystemMessage(content="5살 아이에게 설명하듯이 쉬운 단어를 사용하세요."), question], config=lf_config)
print("[교사]", r2.content[:150])

[과학자] 중력은 **질량을 가진 물체들 사이에 작용하는 끌어당기는 힘**입니다. 아주 간단히 말하면, 지구가 우리를 아래로 끌어당기기 때문에 우리는 바닥에 서 있을 수 있습니다.

핵심만 정리하면:

- **질량이 있으면 중력을 만듭니다.**
- **질량이 클수록 중력이 강합니

[교사] 중력은 **물건을 아래로 끌어당기는 힘**이에요.

예를 들면:
- 사과를 놓으면 **땅으로 떨어져요**
- 우리가 뛰어도 **다시 땅에 내려와요**
- 달도 지구의 중력 때문에 **지구 주변을 빙빙 돌아요**

왜 그럴까요?
- 지구가 아주 커서
- 물건을 **“여기


## 1.4 딕셔너리 형식

메시지 객체 대신 딕셔너리로도 전달할 수 있습니다. LangChain은 메시지 입력을 세 가지 형식으로 지원합니다:

1. **문자열(String)**: 단순 텍스트 프롬프트에 적합 (예: `model.invoke("Hello")`)
2. **메시지 객체(Message objects)**: `SystemMessage`, `HumanMessage` 등 타입이 지정된 인스턴스 리스트
3. **딕셔너리(Dictionary)**: OpenAI Chat Completion API와 동일한 `{"role": ..., "content": ...}` 구조

세 가지 형식 모두 동일한 결과를 반환하므로, 상황에 맞는 형식을 선택하면 됩니다. 딕셔너리 형식은 기존 OpenAI 코드를 LangChain으로 마이그레이션할 때 특히 유용합니다.

In [5]:
response = model.invoke([
    {"role": "system", "content": "한국어로 답변하세요."},
    {"role": "user", "content": "LangChain이란 무엇인가요?"},
], config=lf_config)
print(response.content[:200])

LangChain은 **대규모 언어 모델(LLM)** 을 활용한 애플리케이션을 더 쉽게 만들 수 있도록 도와주는 **오픈소스 프레임워크**입니다.

쉽게 말하면, ChatGPT 같은 모델을 그냥 호출하는 수준을 넘어:

- **문서와 연결**해서 답하게 하거나
- **외부 API**를 사용하게 하거나
- **여러 단계의 작업**을 자동으로 수행하게 하거나



## 1.5 스트리밍

`model.stream()`을 사용하면 토큰이 생성되는 대로 실시간으로 출력됩니다.
사용자 체감 속도가 크게 향상됩니다.

LangChain 모델은 세 가지 호출 방식을 제공합니다:
- **`invoke()`**: 동기 호출로 전체 응답을 한 번에 반환
- **`stream()`**: 토큰 단위로 `AIMessageChunk` 객체를 순차 반환하여 실시간 출력 가능
- **`batch()`**: 여러 요청을 동시에 처리하여 효율성 향상

스트리밍 중에는 각 `AIMessageChunk`가 점진적으로 결합되어 최종 메시지를 구성하며, 토큰 사용량도 점진적으로 추적할 수 있습니다.

In [6]:
print("스트리밍 응답: ", end="")
for chunk in model.stream("우주에 대한 재미있는 사실을 2문장으로 알려주세요.", config=lf_config):
    print(chunk.content, end="", flush=True)
print()  # 줄바꿈

스트리밍 응답: 우주는 계속 팽창하고 있어서, 아주 먼 은하는 우리에게서 점점 더 빠르게 멀어지고 있어요. 또 우주에는 별보다 더 많은 행성이 있을 것으로 추정될 만큼, 아직 우리가 다 알지 못하는 신비한 세계가 가득합니다.


## 1.6 배치 호출

`model.batch()`로 여러 질문을 한 번에 보낼 수 있습니다. 개별적으로 `invoke()`를 반복 호출하는 것보다 효율적으로 여러 요청을 병렬 처리합니다.

In [7]:
responses = model.batch(["2+2는 얼마인가요?", "3*5는 얼마인가요?"], config=lf_config)
for r in responses:
    print("-", r.content[:100])

- 2 + 2는 **4**입니다.
- 3 × 5 = 15입니다.


### 질문: 병렬로 실행되는 거 맞아?

In [8]:
import time
from langchain_core.callbacks import BaseCallbackHandler

class TimingCallback(BaseCallbackHandler):
    def on_llm_start(self, *args, **kwargs):
        print(f"[{time.time():.3f}] LLM 호출 시작")
    def on_llm_end(self, *args, **kwargs):
        print(f"[{time.time():.3f}] LLM 호출 완료")

responses = model.batch(
    ["긴 에세이를 써줘", "2+2는?"],
    config={"callbacks": [TimingCallback()]}
)

[1774310987.242] LLM 호출 시작
[1774310987.246] LLM 호출 시작
[1774310987.992] LLM 호출 완료
[1774310996.725] LLM 호출 완료


### 타임스탬프 분석

| 이벤트 | 시간 | 차이 |
|---|---|---|
| 호출 시작 ① | 311.752 | — |
| 호출 시작 ② | 311.752 | **0.000초** (동시 출발) |
| 호출 완료 ① | 313.602 | — |
| 호출 완료 ② | 313.604 | **0.002초** 차이 |

두 요청이 **동일한 시각에 출발**해서 **0.002초 차이로 거의 동시에 완료**됐어요. 이건 순차 실행이었다면 불가능한 결과입니다. 순차였다면 이런 식이어야 하거든요:

```
시작 → 311.752
완료 → 313.602  (약 1.85초)
시작 → 313.602
완료 → 315.4xx  (또 1.85초)
```

총 ~3.7초가 걸려야 하는데, 실제로는 ~1.85초만에 둘 다 끝났으니 **병렬 처리가 확실히 작동하고 있는 겁니다.**

## 요약

| 개념 | 설명 |
|---|---|
| `SystemMessage` | 모델의 페르소나·규칙 설정 |
| `HumanMessage` | 사용자 입력 |
| `model.invoke()` | 동기 호출 (전체 응답) |
| `model.stream()` | 토큰 단위 실시간 출력 |
| `model.batch()` | 여러 요청 동시 처리 |

### 다음 단계
→ **[02_langchain_basics.ipynb](./02_langchain_basics.ipynb)**: LangChain으로 도구를 갖춘 에이전트를 만듭니다.